# Quantum Canary demo

Quantum Canary checks the health of qubits in one short job.

- **IBM (superconducting):** 17 probe circuits per qubit (8,700 shots) measure T1, T2 (echo),
  T2\* (Ramsey), the frequency offset Δω and the single-qubit gate error ε, each with an error bar,
  a fit check (χ²/dof) and a GREEN / YELLOW / RED verdict.
- **IonQ (trapped ions):** T1 and T2 last seconds, so only ε is measured. The report, saving,
  comparing and monitoring work exactly the same way.

Contents:
1. Settings
2. Check qubits and read the results
3. Use the results in your own code
4. Save and load
5. Compare two checks
6. The command line
7. IonQ trapped ions
8. No hardware: simulate, with the true values

By default everything runs on IBM's **fake** backends: free, local, no account. Each check takes
a minute or two because your laptop simulates it.

## Install

From a clone of the repository, in a terminal with your virtual environment active:
```
python -m pip install -e .
quantum-canary --version
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import math
import pathlib
import subprocess
import sys

import quantum_canary as canary
canary.__version__

## 1. Settings

Edit this cell. Real hardware uses QPU time or credits, so both switches start off.

- `USE_IBM = True` runs on real IBM hardware through your saved account
  (`QiskitRuntimeService.save_account(...)`, once).
- `RUN_IONQ = True` runs section 7 on IonQ. It needs `IONQ_API_KEY` set in your environment.
  `IONQ_BACKEND = "simulator"` is IonQ's free cloud simulator (with the Forte-1 noise model);
  `"qpu.forte-1"` is the real machine and costs credits.

In [ ]:
USE_IBM = False
IBM_BACKEND = "ibm_kingston"      # the fake copy is used when USE_IBM is False
QUBITS = [13, 14]                 # neighbours on Kingston, so they are probed in two groups

RUN_IONQ = False
IONQ_BACKEND = "simulator"
IONS = [0, 3]

OUT = pathlib.Path("demo_output")  # where this notebook saves its files
OUT.mkdir(exist_ok=True)

In [ ]:
if USE_IBM:
    from qiskit_ibm_runtime import QiskitRuntimeService
    backend = QiskitRuntimeService().backend(IBM_BACKEND)
else:
    from qiskit_ibm_runtime.fake_provider import FakeProviderForBackendV2
    backend = FakeProviderForBackendV2().backend("fake_" + IBM_BACKEND.removeprefix("ibm_"))

backend.name, backend.num_qubits

## 2. Check qubits and read the results

`canary.check(backend, qubits)` sends one job, fits the counts and returns a **Report**.

In [ ]:
r = canary.check(backend, QUBITS)
r.print()

How to read the table:

- Each number is `value ± error bar`. Repeated checks scatter by about one error bar.
- **fit**: `ok`, or which probe fit poorly (e.g. `T1 6.2 > 4.4`). Only that probe's numbers are in doubt.
- **health**: GREEN = trust the numbers; YELLOW = a fit was poor or T2\* and T2 differ by more than
  20% (the bracket says which); RED = a fit failed, re-run.
- Health describes the measurement, not the qubit's quality.

Each qubit's results as plain numbers. Times are in seconds and Δω in rad/s:

In [ ]:
q = r[QUBITS[0]]
print(f"T1   = {q.t1 * 1e6:6.1f} ± {q.t1_sigma * 1e6:.1f} µs")
print(f"T2   = {q.t2 * 1e6:6.1f} ± {q.t2_sigma * 1e6:.1f} µs")
print(f"T2*  = {q.t2_star * 1e6:6.1f} ± {q.t2_star_sigma * 1e6:.1f} µs")
print(f"dw   = {q.delta_omega / (2 * math.pi):6.1f} ± {q.delta_omega_sigma / (2 * math.pi):.1f} Hz")
print(f"eps  = {q.eps:.2e} ± {q.eps_sigma:.0e}")
print("health:", q.health, "|", q.health_reason or "-", "| poor fits:", q.fit)
print("advice:", q.dd_advice, q.frame_advice)

## 3. Use the results in your own code

Canary only reports; you decide.

In [ ]:
trusted = [q.qubit for q in r if q.health == "GREEN"]
retry = [q.qubit for q in r if q.health == "RED"]
use_dd = any(q.dd_advice for q in r)

print("trust:", trusted, "| re-run:", retry, "| turn on dynamical decoupling:", use_dd)
for q in r:
    if q.health != "GREEN":
        print(f"qubit {q.qubit} is {q.health}: {q.health_reason} {q.fit}")

## 4. Save and load

The file holds every number, the priors used and the raw shot counts. `canary.load` gives the
Report back; files from the command line load the same way.

In [ ]:
r.save(OUT / "before.json")
before = canary.load(OUT / "before.json")
print(before[QUBITS[0]].t1 == r[QUBITS[0]].t1)

## 5. Compare two checks

`canary.compare(before, after)` asks, per qubit and per number, how many combined error bars the
two checks are apart (z), and flags a change only past a limit corrected for testing four numbers
at once. A qubit that did not change is flagged about 5% of the time.

In [ ]:
after = canary.check(backend, QUBITS)
after.save(OUT / "after.json")

d = canary.compare(before, after)
d.print()
print("changed:", d.changed, "| stable:", d.stable)

## 6. The command line

The same two operations from a terminal:
```
quantum-canary check fake_kingston 13,14 --json a.json
quantum-canary compare a.json b.json
quantum-canary compare first.json second.json third.json     # every later file vs the first
```
Here they run through the notebook's Python (`cli` prints the output with the right encoding).

In [ ]:
def cli(*args):
    out = subprocess.run([sys.executable, "-m", "quantum_canary", *map(str, args)],
                         capture_output=True, text=True, encoding="utf-8")
    print(out.stderr + out.stdout)

qubits_arg = ",".join(map(str, QUBITS))
cli("check", backend.name, qubits_arg, "--json", OUT / "cli.json")

In [ ]:
cli("compare", OUT / "before.json", OUT / "after.json", OUT / "cli.json")

## 7. IonQ trapped ions

The same `canary.check` on a qiskit-ionq backend measures ε on each ion, one ion per circuit.
Only ε is measured, so T1, T2, T2\* and Δω show as `n/a`, and health uses the gate fit alone.
From a terminal: `quantum-canary check qpu.forte-1 0,3` (with `IONQ_API_KEY` set).

`canary.check` has no cost limit. On the real machine use
`canary.run_ionq(backend, ions, max_cost=...)`, which refuses a run whose IonQ estimate is higher.

In [ ]:
if RUN_IONQ:
    import os
    from qiskit_ionq import IonQProvider
    ion_backend = IonQProvider(os.environ["IONQ_API_KEY"]).get_backend(IONQ_BACKEND, gateset="native")
    if IONQ_BACKEND == "simulator":
        ion_backend.set_options(noise_model="forte-1")

    ions_before = canary.check(ion_backend, IONS)
    ions_before.print()
    ions_before.save(OUT / "ions_before.json")

    ions_after = canary.check(ion_backend, IONS)
    canary.compare(ions_before, ions_after).print()
else:
    print("RUN_IONQ is False. Set it in section 1 to run this on IonQ.")

## 8. No hardware: simulate, with the true values

`canary.simulate` invents qubits with hidden true values, measures them with the same probe and
returns the same Report, plus `q.truth`. Same seed, same qubits. `regime="ideal"` follows Canary's
model exactly; `regime="nisq"` (default) adds drift, slow frequency noise and coherent gate error.

In [ ]:
sc = canary.simulate(2, architecture="superconducting", regime="nisq")
sc.print()

In [ ]:
pairs = [("t1", "T1_s"), ("t2", "T2_s"), ("delta_omega", "delta_omega"), ("eps", "epsilon_sx")]

def within_two_sigma(report):
    for q in report:
        for name, key in pairs:
            value, sigma, truth = getattr(q, name), getattr(q, name + "_sigma"), q.truth[key]
            print(f"qubit {q.qubit} {name:12s} {value:10.3g}  true {truth:10.3g}  "
                  f"within 2 error bars: {abs(value - truth) < 2 * sigma}")

within_two_sigma(sc)

ε often lands outside its error bars in the NISQ regime: the true value includes coherent gate
error, which Canary's gate probe cancels by design, so its ε reads low by the coherent share.

Trapped ions, simulated (all five numbers, since a simulation can measure T1 and T2):

In [ ]:
ti = canary.simulate(2, architecture="trapped_ion", regime="nisq")
ti.print()

## Clean up

Everything this notebook wrote is in `demo_output`.

In [ ]:
# import shutil; shutil.rmtree(OUT)